# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [57]:

!git clone https://github.com/maram-elaian/brandora.git

import os
os.chdir('/kaggle/working/brandora')

print(os.getcwd())

!ls -l data/

fatal: destination path 'brandora' already exists and is not an empty directory.
/kaggle/working/brandora
total 16
-rw-r--r-- 1 root root 15771 Sep 20 09:47 test_briefs.json


In [58]:
!pip install -q groq
from groq import Groq
client = Groq(api_key=groq_key)


In [59]:
models = client.models.list()
print(len(models.data))
for i in models.data:
    print(f"- {i.id}")

13
- meta-llama/llama-prompt-guard-2-22m
- allam-2-7b
- openai/gpt-oss-20b
- whisper-large-v3
- openai/gpt-oss-120b
- canopylabs/orpheus-v1-english
- openai/gpt-oss-safeguard-20b
- whisper-large-v3-turbo
- groq/compound-mini
- meta-llama/llama-prompt-guard-2-86m
- groq/compound
- canopylabs/orpheus-arabic-saudi
- qwen/qwen3.8-27b


In [60]:
import json

# فتح ملف الاختبار
with open('data/test_briefs.json', 'r', encoding='utf-8') as file:
    test_briefs = json.load(file)

# اختيار أول حالة فقط (BR001 - مقهى للطلاب)
sample_brief = test_briefs[0]

# عرض البيانات للتأكد
print("✅ تم تحميل البيانات بنجاح!")
print(f"\n📝 حالة الاختبار المختارة:")
print(f"Industry: {sample_brief['industry']}")
print(f"Target Audience: {sample_brief['target_audience']}")
print(f"Purpose: {sample_brief['brand_purpose']}")

✅ تم تحميل البيانات بنجاح!

📝 حالة الاختبار المختارة:
Industry: coffee
Target Audience: university students
Purpose: provide a comfortable and energetic place for studying and socializing


In [61]:
system_prompt = """You are an expert Brand Strategist. 
Your task is to analyze a raw brand brief and extract a structured Brand Specification.
You MUST output ONLY valid JSON. Do not include any markdown formatting (like ```json) or extra text."""


user_prompt = f"""Analyze this brief:
Industry: {sample_brief['industry']}
Target Audience: {sample_brief['target_audience']}
Description: {sample_brief['brand_purpose']}

Return a JSON object with EXACTLY this structure:
{{
  "industry": "string",
  "target_audience": "string",
  "brand_purpose": "string",
  "personality": ["string", "string"],
  "tone": "string",
  "visual_style": ["string", "string"],
  "keywords": ["string", "string"],
  "color_direction": ["string", "string"],
  "logo_direction": "string"
}}"""

print("done")

done


In [62]:
#GPT-OSS-20B
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # اسم النموذج الدقيق من القائمة
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)
raw_output = response.choices[0].message.content
clean_json_string = raw_output.replace("```json", "").replace("```", "").strip()
brand_spec = json.loads(clean_json_string)
result_1 = json.loads(clean_json)
print("✅ نجح GPT-OSS-20B! إليك النتيجة:")
print(json.dumps(brand_spec, indent=2, ensure_ascii=False))

✅ نجح GPT-OSS-20B! إليك النتيجة:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "to create a welcoming, energetic study and social hub for students",
  "personality": [
    "Friendly",
    "Energetic"
  ],
  "tone": "warm and upbeat",
  "visual_style": [
    "Modern minimalism",
    "Cozy urban"
  ],
  "keywords": [
    "study",
    "community"
  ],
  "color_direction": [
    "earthy tones",
    "vibrant pops"
  ],
  "logo_direction": "simple, icon-based with a coffee cup and book"
}


In [63]:
#gpt-oss-120b

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_2 = json.loads(clean_json)

print("✅ النتيجة من openai/gpt-oss-120b:")
print(json.dumps(result_2, indent=2, ensure_ascii=False))

✅ النتيجة من openai/gpt-oss-120b:
{
  "industry": "Coffee",
  "target_audience": "University students",
  "brand_purpose": "Create a comfortable, energetic environment where students can study, collaborate, and socialize over quality coffee.",
  "personality": [
    "Energetic",
    "Welcoming"
  ],
  "tone": "Friendly and motivating",
  "visual_style": [
    "Modern",
    "Cozy"
  ],
  "keywords": [
    "Study",
    "Community",
    "Energy"
  ],
  "color_direction": [
    "Warm neutrals",
    "Vibrant accent colors"
  ],
  "logo_direction": "A simple, versatile mark that combines a coffee cup with a subtle book or campus element."
}


In [64]:
#qwen3.8-27b

response = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_3 = json.loads(clean_json)

print("✅ النتيجة من qwen/qwen3.8-27b:")
print(json.dumps(result_3, indent=2, ensure_ascii=False))

✅ النتيجة من qwen/qwen3.8-27b:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "To serve as a third place that balances academic focus with social connection, providing the energy and comfort needed for student life.",
  "personality": [
    "Energetic",
    "Welcoming"
  ],
  "tone": "Casual and encouraging",
  "visual_style": [
    "Modern minimalism",
    "Warm and inviting"
  ],
  "keywords": [
    "Focus",
    "Community",
    "Energy",
    "Comfort"
  ],
  "color_direction": [
    "Earthy neutrals",
    "Vibrant accents"
  ],
  "logo_direction": "A clean, typographic mark with a subtle iconographic element representing both a coffee cup and a book or spark, designed for high visibility on digital screens and merchandise."
}


#  Comparing 3 models 
---
   - qwen/qwen3.8-27b
   -  openai/gpt-oss-20b
   -  openai/gpt-oss-120b

In [65]:
import pandas as pd

comparison_data = [
    {
        "Model": "GPT-OSS-20B",
        "Personality": ", ".join(result_1.get('personality', [])),
        "Tone": result_1.get('tone', ''),
        "Visual Style": ", ".join(result_1.get('visual_style', [])),
        "Color Direction": ", ".join(result_1.get('color_direction', [])),
        "Keywords": ", ".join(result_1.get('keywords', [])),
        "Logo Direction": result_1.get('logo_direction', '')
    },
    {
        "Model": "GPT-OSS-120B",
        "Personality": ", ".join(result_2.get('personality', [])),
        "Tone": result_2.get('tone', ''),
        "Visual Style": ", ".join(result_2.get('visual_style', [])),
        "Color Direction": ", ".join(result_2.get('color_direction', [])),
        "Keywords": ", ".join(result_2.get('keywords', [])),
        "Logo Direction": result_2.get('logo_direction', '')
    },
    {
        "Model": "Qwen 3.8 27B",
        "Personality": ", ".join(result_3.get('personality', [])),
        "Tone": result_3.get('tone', ''),
        "Visual Style": ", ".join(result_3.get('visual_style', [])),
        "Color Direction": ", ".join(result_3.get('color_direction', [])),
        "Keywords": ", ".join(result_3.get('keywords', [])),
        "Logo Direction": result_3.get('logo_direction', '')
    }
]


df_full_comparison = pd.DataFrame(comparison_data)



display(df_full_comparison)

,Model,Personality,Tone,Visual Style,Color Direction,Keywords,Logo Direction
0,GPT-OSS-20B,"Energetic, Welcoming",Friendly and inspiring,"Modern minimalist, Playful","Warm earth tones, Vibrant accents","Focus, Community","A clean, modern mark that subtly integrates el..."
1,GPT-OSS-120B,"Energetic, Welcoming",Friendly and motivating,"Modern, Cozy","Warm neutrals, Vibrant accent colors","Study, Community, Energy","A simple, versatile mark that combines a coffe..."
2,Qwen 3.8 27B,"Energetic, Welcoming",Casual and encouraging,"Modern minimalism, Warm and inviting","Earthy neutrals, Vibrant accents","Focus, Community, Energy, Comfort","A clean, typographic mark with a subtle iconog..."
